# 06 - Interactive Research Dashboard

**Deep Learning IDS Robustness Against LLM-Generated Polymorphic Attacks**

This notebook launches a Streamlit dashboard built on top of the project folder on Google Drive
(`/content/drive/MyDrive/IDS_Polymorphic_Project`). It retrains nothing: it reads the configs, processed
data, saved checkpoints and result CSVs written by notebooks 01-05 and runs live inference with the saved models.

| Tab | Content |
|---|---|
| 1. System Pipeline & Overview | Interactive 5-stage architecture flowchart and executive KPI cards |
| 2. Feature-Space Projection (PCA & UMAP) | Benign vs original attack vs LLM variants on log1p mutable features |
| 3. Model Benchmark (E1-E4) | Per-class / per-regime comparison across RandomForest, MLP, CNN1D, LSTM |
| 4. Live Evasion & Defense Simulator | Original flow, LLM feature shift, baseline vs retrained verdicts |

**Preprocessing used for inference (checked against notebooks 01, 02, 04 and 05):** every model, RandomForest
included, receives `scaler.transform(raw 59 features)` with the `StandardScaler` saved in
`data/processed/scaler.joblib`. `log1p` is applied only in the projection tab, as in the notebooks' own figures.
Checkpoints are loaded with `ids_common.load_model` from `models/baseline/` (E1) and `models/retrained_{20,50}pct/` (E4).

Run the three code cells in order. Cell 3 prints a public `trycloudflare.com` link and opens it in a new tab.

## Cell 1 - Mount Google Drive, install dependencies, download cloudflared

Mounts Drive, checks the project folder, installs `streamlit`, `plotly` and `umap-learn`, downloads the
`cloudflared` binary for the public tunnel, and prints an inventory of the files the dashboard reads.
Items marked `optional` only enable extra panels; the app shows a status indicator instead of failing when they are absent.

In [ ]:
import os, sys, glob, json, subprocess
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
assert os.path.isdir(BASE_DIR), f'Project folder not found: {BASE_DIR}'
os.environ['IDS_BASE_DIR'] = BASE_DIR
os.chdir('/content')
print('Project folder:', BASE_DIR)

!pip install -q streamlit plotly umap-learn

CLOUDFLARED = '/content/cloudflared'
if not os.path.exists(CLOUDFLARED):
    subprocess.run(['wget', '-q', '-O', CLOUDFLARED,
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
                   check=True)
subprocess.run(['chmod', '+x', CLOUDFLARED], check=True)
print(subprocess.run([CLOUDFLARED, '--version'], capture_output=True, text=True).stdout.strip())

# ------------------------------------------------------------------ inventory
PCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'project_config.json')))
DIRS = {k: v.replace(PCFG['dirs']['base'], BASE_DIR) for k, v in PCFG['dirs'].items()}

def size(p):
    if os.path.isdir(p):
        return f'{len(os.listdir(p))} items'
    return f'{os.path.getsize(p) / 1024 ** 2:.2f} MB'

def report(title, items):
    print(f'\n{title}')
    print('-' * 78)
    for rel, path, required in items:
        ok = os.path.exists(path)
        tag = 'OK     ' if ok else ('MISSING' if required else 'absent ')
        note = '' if required else '  (optional)'
        print(f'  [{tag}] {rel:<52} {size(path) if ok else "":>12}{note}')

report('Configuration and shared code', [
    (f'config/{f}', os.path.join(BASE_DIR, 'config', f), f != 'variant_generation_config.json')
    for f in ['project_config.json', 'feature_config.json', 'variant_generation_config.json']] +
    [('code/ids_common.py', os.path.join(BASE_DIR, 'code', 'ids_common.py'), True)])

report('Processed data', [(f'data/processed/{f}', os.path.join(DIRS['processed'], f), f in ('train_raw.parquet', 'scaler.joblib'))
                          for f in ['train_raw.parquet', 'scaler.joblib', 'X_test.npy', 'y_test.npy']])

report('Variants', [
    ('data/variants/llm_variants.csv', os.path.join(DIRS['variants'], 'llm_variants.csv'), True),
    ('data/variants/scripted_variants.csv', os.path.join(DIRS['variants'], 'scripted_variants.csv'), False),
    ('data/variants/validated/all_validated_variants.parquet',
     os.path.join(DIRS['variants'], 'validated', 'all_validated_variants.parquet'), False),
    ('data/variants/variant_train_pool.parquet', os.path.join(DIRS['variants'], 'variant_train_pool.parquet'), False),
    ('data/variants/variant_test_heldout.parquet', os.path.join(DIRS['variants'], 'variant_test_heldout.parquet'), False)])

print('\nModel checkpoints')
print('-' * 78)
model_dirs = [os.path.join(DIRS['models'], 'baseline')] + sorted(glob.glob(os.path.join(DIRS['models'], 'retrained_*pct')))
for d in model_dirs:
    files = sorted(os.path.basename(f) for f in glob.glob(os.path.join(d, '*_seed*.*')))
    print(f'  {os.path.relpath(d, BASE_DIR):<28} {len(files):>3} checkpoint(s)')
    for m in ['RandomForest', 'MLP', 'CNN1D', 'LSTM']:
        seeds = [f.split('_seed')[1].split('.')[0] for f in files if f.startswith(m + '_seed')]
        print(f'      {m:<14} seeds: {", ".join(seeds) if seeds else "none"}')

csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DIRS['results'], '*.csv')))
figs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DIRS['results'], 'figures', '*.png')))
print(f'\nResult CSVs ({len(csvs)})')
print('-' * 78)
for f in csvs:
    print('  ' + f)
print(f'\nFigures in results/figures ({len(figs)})')
print('-' * 78)
for f in figs:
    print('  ' + f)

## Cell 2 - Generate `app.py`

Writes the Streamlit application to `/content/app.py`. It imports the exact model classes and loaders from
`code/ids_common.py` on Drive, caches checkpoints with `st.cache_resource` and tables with `st.cache_data`,
and keeps a checkpoint status panel in the sidebar.

In [ ]:
%%writefile app.py
"""Deep Learning IDS Robustness Against LLM-Generated Polymorphic Attacks - interactive dashboard.

Generated by 06_interactive_dashboard.ipynb. Every number shown is read from the project folder on
Google Drive (configs, processed data, saved checkpoints, result CSVs) or computed live from them.
Preprocessing matches notebooks 01/02/04/05 exactly: all four models (RandomForest included) consume
scaler.transform(raw 59 features) with the StandardScaler fitted in notebook 01. log1p is used only
for the feature-space projections, exactly as in the notebooks' own visualisations.
"""
import os
import re
import sys
import glob
import json
import html

import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go

BASE_DIR = os.environ.get('IDS_BASE_DIR', '/content/drive/MyDrive/IDS_Polymorphic_Project')
APP_TITLE = 'Deep Learning IDS Robustness Against LLM-Generated Polymorphic Attacks'
PM = ' \u00b1 '      # plus-minus sign, kept as an escape so this source file stays pure ASCII
EN_DASH = '\u2013'

st.set_page_config(page_title='IDS Polymorphic Robustness', layout='wide', initial_sidebar_state='expanded')

# ------------------------------------------------------------------ palette
NAVY, STEEL, TEAL, AMBER, CRIMSON = '#1E3A8A', '#0284C7', '#0D9488', '#D97706', '#DC2626'
OK_GREEN, BAD_RED = '#10B981', '#EF4444'
SLATE, INK, MUTED, GRID, BORDER = '#94A3B8', '#0F172A', '#64748B', '#E5E7EB', '#E2E8F0'
MODEL_ORDER = ['RandomForest', 'MLP', 'CNN1D', 'LSTM']
MODEL_COLORS = {'RandomForest': NAVY, 'MLP': STEEL, 'CNN1D': TEAL, 'LSTM': AMBER}
PROFILE_ORDER = ['extended_duration_low_rate', 'high_variance_clustered', 'reduced_magnitude', 'mixed_shift']
PROFILE_COLORS = dict(zip(PROFILE_ORDER, [STEEL, TEAL, AMBER, CRIMSON]))
REGIME_COLORS = {'E1': NAVY, 'E2': CRIMSON, 'E3': AMBER, 'E4@20': STEEL, 'E4@50': TEAL}
KEY_FEATURES = ['Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean', 'Flow IAT Mean',
                'Flow IAT Std', 'Flow IAT Max', 'Total Fwd Packets', 'Total Backward Packets',
                'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Active Mean', 'Idle Mean']

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
html, body, [class*="css"], .stMarkdown, .stText { font-family: 'Inter', system-ui, sans-serif; }
.block-container { padding-top: 1.6rem; padding-bottom: 3rem; max-width: 1500px; }
h1, h2, h3, h4 { color: #0F172A; letter-spacing: -0.01em; }
.hero { border: 1px solid #E2E8F0; border-left: 5px solid #1E3A8A; border-radius: 12px;
        padding: 18px 24px; background: linear-gradient(90deg, #F8FAFC 0%, #FFFFFF 70%); margin-bottom: 14px; }
.hero-title { font-size: 1.55rem; font-weight: 700; color: #0F172A; margin: 0; line-height: 1.25; }
.hero-sub { color: #64748B; font-size: 0.92rem; margin-top: 6px; }
.section-label { text-transform: uppercase; letter-spacing: 0.08em; font-size: 0.72rem; font-weight: 600;
                 color: #64748B; margin: 18px 0 6px 0; }
.kpi-card { border: 1px solid #E2E8F0; border-radius: 10px; padding: 14px 16px; background: #FFFFFF;
            box-shadow: 0 1px 2px rgba(15, 23, 42, 0.04); height: 100%; }
.kpi-label { font-size: 0.72rem; text-transform: uppercase; letter-spacing: 0.07em; color: #64748B; font-weight: 600; }
.kpi-value { font-size: 1.65rem; font-weight: 700; color: #0F172A; margin-top: 4px; line-height: 1.15; }
.kpi-sub { font-size: 0.80rem; color: #64748B; margin-top: 4px; }
.pill { display: inline-block; padding: 2px 10px; border-radius: 999px; font-size: 0.68rem; font-weight: 700;
        letter-spacing: 0.07em; text-transform: uppercase; vertical-align: middle; white-space: nowrap; }
.pill-ok  { background: rgba(16, 185, 129, 0.12); color: #047857; border: 1px solid #10B981; }
.pill-bad { background: rgba(239, 68, 68, 0.10); color: #B91C1C; border: 1px solid #EF4444; }
.pill-neutral { background: #F1F5F9; color: #475569; border: 1px solid #CBD5E1; }
.model-card { border: 1px solid #E2E8F0; border-radius: 10px; padding: 12px 14px; background: #FFFFFF;
              margin-bottom: 10px; }
.model-card.ok  { border-left: 4px solid #10B981; }
.model-card.bad { border-left: 4px solid #EF4444; }
.model-card.neutral { border-left: 4px solid #CBD5E1; }
.mc-head { display: flex; justify-content: space-between; align-items: center; gap: 8px; }
.mc-name { font-weight: 700; color: #0F172A; font-size: 0.95rem; }
.mc-row { font-size: 0.82rem; color: #475569; margin-top: 6px; }
.mc-row b { color: #0F172A; }
.conf-track { height: 6px; background: #F1F5F9; border-radius: 4px; margin-top: 8px; overflow: hidden; }
.conf-fill { height: 6px; border-radius: 4px; }
.panel-title { font-weight: 700; font-size: 1.0rem; color: #0F172A; margin-bottom: 2px; }
.panel-sub { font-size: 0.80rem; color: #64748B; margin-bottom: 10px; }
.meta-strip { display: flex; flex-wrap: wrap; gap: 18px; border: 1px solid #E2E8F0; border-radius: 10px;
              padding: 10px 16px; background: #F8FAFC; font-size: 0.82rem; color: #475569; margin: 6px 0 10px 0; }
.meta-strip b { color: #0F172A; }
.callout { border: 1px solid #E2E8F0; border-left: 4px solid #0284C7; border-radius: 8px; padding: 10px 14px;
           background: #F8FAFC; font-size: 0.86rem; color: #334155; margin: 8px 0; }
.status-row { display: flex; justify-content: space-between; align-items: center; font-size: 0.82rem;
              padding: 3px 0; border-bottom: 1px dashed #E2E8F0; }
.stTabs [data-baseweb="tab-list"] { gap: 4px; border-bottom: 1px solid #E2E8F0; }
.stTabs [data-baseweb="tab"] { padding: 10px 16px; font-weight: 600; color: #475569; }
.stTabs [aria-selected="true"] { color: #1E3A8A; }
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


# ------------------------------------------------------------------ small helpers
def esc(x):
    return html.escape(str(x))


def pill(text, kind):
    return f'<span class="pill pill-{kind}">{esc(text)}</span>'


def rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'


def fmt_pct(x, d=1):
    return 'n/a' if x is None or not np.isfinite(x) else f'{100 * x:.{d}f}%'


def fmt_num(x, d=3, signed=False):
    if x is None or not np.isfinite(x):
        return 'n/a'
    return f'{x:+.{d}f}' if signed else f'{x:.{d}f}'


def parse_pm(cell):
    """'0.792 +- 0.012' style cell -> (mean, std)."""
    nums = re.findall(r'-?\d+(?:\.\d+)?(?:[eE]-?\d+)?', str(cell))
    if not nums:
        return np.nan, np.nan
    return float(nums[0]), (float(nums[1]) if len(nums) > 1 else 0.0)


def style_fig(fig, height=420, legend_title=None):
    fig.update_layout(
        height=height, template='plotly_white', font=dict(family='Inter, system-ui, sans-serif', size=12, color=INK),
        margin=dict(l=10, r=10, t=40, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='#FFFFFF',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0, title=legend_title,
                    font=dict(size=11)),
        hoverlabel=dict(bgcolor='#FFFFFF', bordercolor=BORDER, font=dict(color=INK)))
    fig.update_xaxes(showgrid=False, linecolor=BORDER, ticks='outside', tickcolor=BORDER)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=BORDER)
    return fig


def kpi_card(label, value, sub, accent):
    return (f'<div class="kpi-card" style="border-top:3px solid {accent}"><div class="kpi-label">{esc(label)}</div>'
            f'<div class="kpi-value">{esc(value)}</div><div class="kpi-sub">{esc(sub)}</div></div>')


def section(label):
    st.markdown(f'<div class="section-label">{esc(label)}</div>', unsafe_allow_html=True)


# ------------------------------------------------------------------ configuration
def _read_json(path):
    with open(path) as fh:
        return json.load(fh)


@st.cache_resource(show_spinner=False)
def load_project(base_dir):
    pcfg = _read_json(os.path.join(base_dir, 'config', 'project_config.json'))
    fcfg = _read_json(os.path.join(base_dir, 'config', 'feature_config.json'))
    vpath = os.path.join(base_dir, 'config', 'variant_generation_config.json')
    vcfg = _read_json(vpath) if os.path.exists(vpath) else {}
    dirs = {k: v.replace(pcfg['dirs']['base'], base_dir) for k, v in pcfg['dirs'].items()}
    dirs['code'] = os.path.join(base_dir, 'code')
    dirs['baseline_models'] = os.path.join(dirs['models'], 'baseline')
    dirs['validated'] = os.path.join(dirs['variants'], 'validated')
    dirs['figures'] = os.path.join(dirs['results'], 'figures')
    return pcfg, fcfg, vcfg, dirs


try:
    PCFG, FCFG, VCFG, DIRS = load_project(BASE_DIR)
except Exception as exc:  # noqa: BLE001
    st.error(f'Could not read the project configuration under {BASE_DIR}/config ({exc}). '
             'Mount Google Drive and run notebooks 01-05 first.')
    st.stop()

CLASS_NAMES = PCFG['class_names']
BENIGN = PCFG['benign_class']
BENIGN_ID = PCFG['benign_id']
ATTACK_CLASSES = PCFG['attack_classes']
LABEL_TO_ID = PCFG['label_to_id']
MODEL_SEEDS = PCFG['model_seeds']
FEATURES = FCFG['feature_names']
MUTABLE = FCFG['mutable_features']
FIXED = FCFG['fixed_features']
PROFILE_TEXT = (VCFG.get('llm', {}) or {}).get('profiles', {})


@st.cache_resource(show_spinner=False)
def load_ids_common(code_dir):
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)
    import ids_common  # the exact module used by notebooks 02, 04 and 05
    return ids_common


try:
    IC = load_ids_common(DIRS['code'])
    IC_ERR = None
except Exception as exc:  # noqa: BLE001
    IC, IC_ERR = None, str(exc)


@st.cache_resource(show_spinner=False)
def get_device():
    return IC.get_device() if IC is not None else None


# ------------------------------------------------------------------ models
def discover_retrained_dirs():
    out = {}
    for d in sorted(glob.glob(os.path.join(DIRS['models'], 'retrained_*pct'))):
        m = re.search(r'retrained_(\d+)pct$', d.rstrip('/'))
        if m and os.path.isdir(d):
            out[int(m.group(1))] = d
    return out


def discover_seeds(model_dir):
    seeds = set()
    for f in glob.glob(os.path.join(model_dir, '*_seed*.*')):
        m = re.search(r'_seed(\d+)\.(pt|joblib)$', f)
        if m:
            seeds.add(int(m.group(1)))
    return sorted(seeds)


@st.cache_resource(show_spinner='Loading model checkpoints from Google Drive ...')
def load_model_set(model_dir, seed):
    models, status = {}, {}
    if IC is None:
        return models, {n: 'ids_common unavailable' for n in MODEL_ORDER}
    for name in MODEL_ORDER:
        path = IC.model_path(model_dir, name, seed)
        if not os.path.exists(path):
            status[name] = 'missing'
            continue
        try:
            models[name] = IC.load_model(name, path, device=get_device())
            status[name] = 'loaded'
        except Exception as exc:  # noqa: BLE001
            status[name] = f'error: {exc}'
    return models, status


def predict_proba(name, model, X):
    """Class-probability matrix (n, n_classes) with columns aligned to CLASS_NAMES ids."""
    if name == 'RandomForest':
        proba = model.predict_proba(X)
        full = np.zeros((len(X), len(CLASS_NAMES)), dtype=np.float64)
        full[:, np.asarray(model.classes_).astype(int)] = proba
        return full
    return IC.predict_proba_dl(model, X, device=get_device())


# ------------------------------------------------------------------ data
@st.cache_resource(show_spinner='Loading training flows (train_raw.parquet) ...')
def load_train_raw():
    p = os.path.join(DIRS['processed'], 'train_raw.parquet')
    if not os.path.exists(p):
        return None
    df = pd.read_parquet(p)
    df['Label'] = df['Label'].astype(str)
    return df.reset_index(drop=True)


@st.cache_resource(show_spinner=False)
def load_scaler():
    p = os.path.join(DIRS['processed'], 'scaler.joblib')
    if not os.path.exists(p):
        return None
    import joblib
    return joblib.load(p)


@st.cache_resource(show_spinner='Loading LLM polymorphic variants ...')
def load_llm_variants():
    df, src = None, None
    vp = os.path.join(DIRS['validated'], 'all_validated_variants.parquet')
    if os.path.exists(vp):
        try:
            d = pd.read_parquet(vp)
            if 'source' in d.columns:
                d = d[d['source'].astype(str).str.lower() == 'llm']
            if len(d):
                df, src = d, 'variants/validated/all_validated_variants.parquet (validated, as evaluated)'
        except Exception:  # noqa: BLE001
            df = None
    if df is None:
        p = os.path.join(DIRS['variants'], 'llm_variants.csv')
        if os.path.exists(p):
            d = pd.read_csv(p, low_memory=False)
            d.columns = d.columns.str.strip()
            df, src = d, 'variants/llm_variants.csv'
    if df is None:
        return None, 'not found'
    df = df.copy()
    for c in FEATURES:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=FEATURES)
    df['Label'] = df['Label'].astype(str).str.strip()
    df = df[df['Label'].isin(ATTACK_CLASSES)]
    df['style'] = df['style'].fillna('unspecified').astype(str) if 'style' in df.columns else 'unspecified'
    if 'source_row_id' in df.columns:
        df['source_row_id'] = pd.to_numeric(df['source_row_id'], errors='coerce')
    else:
        df['source_row_id'] = np.nan
    df['split'] = 'not determined'
    if 'variant_id' in df.columns:
        for fname, tag in [('variant_train_pool.parquet', 'retraining pool'),
                           ('variant_test_heldout.parquet', 'held-out test')]:
            p = os.path.join(DIRS['variants'], fname)
            if os.path.exists(p):
                try:
                    ids = set(pd.read_parquet(p, columns=['variant_id'])['variant_id'])
                    df.loc[df['variant_id'].isin(ids), 'split'] = tag
                except Exception:  # noqa: BLE001
                    pass
    return df.reset_index(drop=True), src


@st.cache_resource(show_spinner=False)
def load_augmentation_selection():
    p = os.path.join(DIRS['results'], '05_augmentation_selection.json')
    if not os.path.exists(p):
        return {}
    try:
        sel = _read_json(p)
        return {k: set(v.get('variant_ids', [])) for k, v in sel.items()}
    except Exception:  # noqa: BLE001
        return {}


@st.cache_data(show_spinner=False)
def read_csv(path, mtime=None):
    return pd.read_csv(path)


def results_csv(name):
    p = os.path.join(DIRS['results'], name)
    return read_csv(p, os.path.getmtime(p)) if os.path.exists(p) else None


def results_stamp():
    files = glob.glob(os.path.join(DIRS['results'], '*.csv'))
    return tuple(sorted((os.path.basename(f), os.path.getmtime(f)) for f in files))


def load_final_table():
    df = results_csv('FINAL_summary_table.csv')
    if df is None:
        return None
    if 'Model' in df.columns:
        df = df.set_index('Model')
    return df


def load_validation_summary():
    for name in ['04_validation_summary.csv', '03_validation_summary.csv']:
        df = results_csv(name)
        if df is None:
            continue
        if 'source' not in df.columns:
            df = df.rename(columns={df.columns[0]: 'source'})
        row = df[df['source'].astype(str).str.lower() == 'llm']
        if len(row) and {'input_rows', 'kept_after_validation'} <= set(df.columns):
            r = row.iloc[0]
            return int(r['input_rows']), int(r['kept_after_validation'])
    return None


@st.cache_data(show_spinner=False)
def load_preprocessing_report(_stamp=None):
    p = os.path.join(DIRS['results'], '01_preprocessing_report.json')
    return _read_json(p) if os.path.exists(p) else {}


# ------------------------------------------------------------------ benchmark table (E1-E4)
@st.cache_data(show_spinner=False)
def build_benchmark(results_dir, _stamp):
    rows, sources = [], []

    def push_stat(regime, model, cls, metric, mean, std, n=None):
        if mean is None or not np.isfinite(mean):
            return
        rows.append({'regime': regime, 'model': str(model), 'class': cls, 'metric': metric,
                     'mean': float(mean), 'std': float(std) if std is not None and np.isfinite(std) else 0.0,
                     'n_seeds': n})

    def push(regime, model, cls, metric, values):
        if values is None:
            return
        v = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
        if len(v):
            push_stat(regime, model, cls, metric, v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0, len(v))

    def tag_for(share):
        return 'E1' if share == 0 else f'E4@{int(round(share * 100))}'

    p05 = os.path.join(results_dir, '05_E4_runs_all_seeds.csv')
    if os.path.exists(p05):
        R = pd.read_csv(p05)
        sources.append('05_E4_runs_all_seeds.csv')
        for (share, model), g in R.groupby(['share', 'model']):
            share = float(share)
            sets = ([('E1', 'original_attacks'), ('E2', 'heldout_llm'), ('E3', 'scripted')] if share == 0
                    else [(tag_for(share), 'heldout_llm')])
            for regime, pre in sets:
                if f'{pre}_recall' not in g.columns:
                    continue
                push(regime, model, 'ALL', 'recall', g[f'{pre}_recall'])
                push(regime, model, 'ALL', 'evasion', g.get(f'{pre}_evasion'))
                for c in ATTACK_CLASSES:
                    push(regime, model, c, 'recall', g.get(f'{pre}_recall_{c}'))
                    push(regime, model, c, 'evasion', g.get(f'{pre}_evasion_{c}'))
            tag = tag_for(share)
            push(tag, model, 'ALL', 'f1_macro', g.get('test_f1_macro'))
            push(tag, model, 'ALL', 'fpr', g.get('test_fpr'))
            if share > 0:
                push(tag, model, 'ALL', 'recovery', g.get('recovery'))
                for c in ATTACK_CLASSES:
                    push(tag, model, c, 'recovery', g.get(f'delta_heldout_llm_recall_{c}'))
    else:
        p02 = os.path.join(results_dir, '02_E1_per_class_mean_std.csv')
        if os.path.exists(p02):
            d = pd.read_csv(p02)
            sources.append('02_E1_per_class_mean_std.csv')
            d = d[(d['metric'] == 'recall') & d['class'].isin(ATTACK_CLASSES)]
            for _, r in d.iterrows():
                push_stat('E1', r['model'], r['class'], 'recall', r['mean'], r['std'])
            for m, g in d.groupby('model'):
                push_stat('E1', m, 'ALL', 'recall', g['mean'].mean(), np.nan)
        p02s = os.path.join(results_dir, '02_E1_summary_mean_std.csv')
        if os.path.exists(p02s):
            d = pd.read_csv(p02s)
            sources.append('02_E1_summary_mean_std.csv')
            d = d.rename(columns={d.columns[0]: 'model'})
            for _, r in d.iterrows():
                for met in ['f1_macro', 'fpr']:
                    if f'{met}_mean' in d.columns:
                        push_stat('E1', r['model'], 'ALL', met, r[f'{met}_mean'], r.get(f'{met}_std', 0.0))
        p04 = os.path.join(results_dir, '04_robustness_summary.csv')
        if os.path.exists(p04):
            d = pd.read_csv(p04)
            sources.append('04_robustness_summary.csv')
            set_map = {'E2_llm': 'E2', 'E3_scripted': 'E3'}
            for _, r in d.iterrows():
                reg = set_map.get(str(r['set']))
                if reg is None:
                    continue
                push_stat(reg, r['model'], r['class'], 'recall', r['variant_recall_mean'], r.get('variant_recall_std', 0))
                push_stat(reg, r['model'], r['class'], 'evasion', r['evasion_mean'], r.get('evasion_std', 0))
        p05r = os.path.join(results_dir, '05_E4_recovery_per_class.csv')
        if os.path.exists(p05r):
            d = pd.read_csv(p05r)
            sources.append('05_E4_recovery_per_class.csv')
            for (share, model), g in d.groupby(['share', 'model']):
                tag = tag_for(float(share))
                if tag == 'E1':
                    continue
                for _, r in g.iterrows():
                    push_stat(tag, model, r['class'], 'recall', r['heldout_recall_mean'], np.nan)
                    push_stat(tag, model, r['class'], 'evasion', r['evasion_mean'], np.nan)
                    push_stat(tag, model, r['class'], 'recovery', r['recovery_mean'], r.get('recovery_std', 0))
                push_stat(tag, model, 'ALL', 'recall', g['heldout_recall_mean'].mean(), np.nan)
                push_stat(tag, model, 'ALL', 'evasion', g['evasion_mean'].mean(), np.nan)

    if not rows:
        pf = os.path.join(results_dir, 'FINAL_summary_table.csv')
        if os.path.exists(pf):
            F = pd.read_csv(pf).set_index('Model')
            sources.append('FINAL_summary_table.csv (macro only)')
            colmap = [('E1', 'recall', 'E1 attack recall'), ('E1', 'f1_macro', 'E1 F1-macro'), ('E1', 'fpr', 'E1 FPR'),
                      ('E2', 'recall', 'E2 recall'), ('E2', 'evasion', 'E2 evasion'), ('E3', 'evasion', 'E3 evasion')]
            for pct in (20, 50):
                colmap += [(f'E4@{pct}', 'recall', f'E4@{pct}% held-out recall'),
                           (f'E4@{pct}', 'recovery', f'E4@{pct}% recovery')]
            for m, r in F.iterrows():
                for reg, met, col in colmap:
                    if col in F.columns:
                        mu, sd = parse_pm(r[col])
                        push_stat(reg, m, 'ALL', met, mu, sd)
                if 'E1 attack recall' in F.columns and 'E3 recall drop' in F.columns:
                    e1, _ = parse_pm(r['E1 attack recall'])
                    d3, s3 = parse_pm(r['E3 recall drop'])
                    push_stat('E3', m, 'ALL', 'recall', e1 - d3, s3)
    return pd.DataFrame(rows), sources


def regime_label(key):
    if key == 'E1':
        return 'E1: Clean Baseline'
    if key == 'E2':
        return 'E2: LLM Polymorphic Variants'
    if key == 'E3':
        return 'E3: Scripted Noise Control'
    m = re.match(r'E4@(\d+)', key)
    return f'E4 @ {m.group(1)}%: Adversarial Retraining' if m else key


def regime_sort_key(key):
    m = re.match(r'E4@(\d+)', key)
    return (4, int(m.group(1))) if m else (int(key[1]), 0)


# ------------------------------------------------------------------ sidebar
BASELINE_SEEDS = discover_seeds(DIRS['baseline_models']) if os.path.isdir(DIRS['baseline_models']) else []
RETRAINED_DIRS = discover_retrained_dirs()

with st.sidebar:
    st.markdown('### Control Panel')
    seed_opts = BASELINE_SEEDS or MODEL_SEEDS
    SEED = st.selectbox('Checkpoint seed', seed_opts, index=seed_opts.index(42) if 42 in seed_opts else 0,
                        help='Seed of the saved checkpoints used for live inference (tabs 4).')
    if RETRAINED_DIRS:
        shares = sorted(RETRAINED_DIRS, reverse=True)
        default_share = 50 if 50 in shares else (20 if 20 in shares else shares[0])
        SHARE = st.selectbox('Retrained models (E4 share)', shares, index=shares.index(default_share),
                             format_func=lambda s: f'{s}% of variant pool added')
        RETRAINED_DIR = RETRAINED_DIRS[SHARE]
    else:
        SHARE, RETRAINED_DIR = None, None

    BASE_MODELS, BASE_STATUS = load_model_set(DIRS['baseline_models'], SEED) if IC else ({}, {})
    RET_MODELS, RET_STATUS = (load_model_set(RETRAINED_DIR, SEED) if (IC and RETRAINED_DIR) else ({}, {}))

    def status_block(title, status):
        st.markdown(f'<div class="section-label">{esc(title)}</div>', unsafe_allow_html=True)
        out = []
        for name in MODEL_ORDER:
            s = status.get(name, 'missing')
            kind, txt = ('ok', 'LOADED') if s == 'loaded' else (('bad', 'ERROR') if s.startswith('error') else ('neutral', 'MISSING'))
            out.append(f'<div class="status-row"><span>{esc(name)}</span>{pill(txt, kind)}</div>')
        st.markdown(''.join(out), unsafe_allow_html=True)
        errs = {k: v for k, v in status.items() if str(v).startswith('error')}
        for k, v in errs.items():
            st.caption(f'{k}: {v[:160]}')

    status_block(f'Baseline checkpoints (E1, seed {SEED})', BASE_STATUS)
    status_block(f'Retrained checkpoints (E4 @ {SHARE}%, seed {SEED})' if SHARE else 'Retrained checkpoints (E4)',
                 RET_STATUS if RET_STATUS else {})

    TRAIN_RAW = load_train_raw()
    SCALER = load_scaler()
    VARIANTS, VARIANT_SRC = load_llm_variants()
    n_csv = len(glob.glob(os.path.join(DIRS['results'], '*.csv')))
    assets = [('train_raw.parquet', TRAIN_RAW is not None, f'{len(TRAIN_RAW):,} rows' if TRAIN_RAW is not None else ''),
              ('scaler.joblib', SCALER is not None, ''),
              ('LLM variants', VARIANTS is not None, f'{len(VARIANTS):,} rows' if VARIANTS is not None else ''),
              ('Result CSVs', n_csv > 0, f'{n_csv} files')]
    st.markdown('<div class="section-label">Data assets</div>', unsafe_allow_html=True)
    st.markdown(''.join(f'<div class="status-row"><span>{esc(a)} <span style="color:#94A3B8">{esc(d)}</span></span>'
                        f'{pill("OK" if ok else "MISSING", "ok" if ok else "neutral")}</div>' for a, ok, d in assets),
                unsafe_allow_html=True)
    if IC_ERR:
        st.caption(f'ids_common could not be imported from {DIRS["code"]}: {IC_ERR[:200]}')
    st.caption(f'Project folder: {BASE_DIR}')

# ------------------------------------------------------------------ header
st.markdown(f'<div class="hero"><div class="hero-title">{esc(APP_TITLE)}</div>'
            '<div class="hero-sub">CIC-IDS2017 &nbsp;|&nbsp; RandomForest, MLP, 1D-CNN, LSTM &nbsp;|&nbsp; '
            'Zero-shot robustness to LLM-crafted flow variants and recovery through adversarial retraining</div></div>',
            unsafe_allow_html=True)

tab1, tab2, tab3, tab4 = st.tabs(['1. System Pipeline & Overview', '2. Feature-Space Projection (PCA & UMAP)',
                                  f'3. Model Benchmark (E1{EN_DASH}E4)', '4. Live Evasion & Defense Simulator'])

FINAL = load_final_table()
BENCH, BENCH_SOURCES = build_benchmark(DIRS['results'], results_stamp())


def bench_value(regime, model, metric, cls='ALL'):
    if BENCH.empty:
        return np.nan
    r = BENCH[(BENCH['regime'] == regime) & (BENCH['model'] == model) & (BENCH['metric'] == metric) & (BENCH['class'] == cls)]
    return float(r['mean'].iloc[0]) if len(r) else np.nan


# ================================================================== TAB 1
def pipeline_figure(stages):
    fig = go.Figure()
    colors = [NAVY, STEEL, TEAL, AMBER, CRIMSON]
    w = 0.86
    n = len(stages)
    for i, s in enumerate(stages):
        col = colors[i % len(colors)]
        fig.add_shape(type='rect', x0=i - w / 2, x1=i + w / 2, y0=0, y1=1, line=dict(color=col, width=1.4),
                      fillcolor=rgba(col, 0.05), layer='below')
        fig.add_shape(type='rect', x0=i - w / 2, x1=i + w / 2, y0=0.80, y1=1, line=dict(width=0),
                      fillcolor=col, layer='below')
        fig.add_annotation(x=i, y=0.935, text=f'<b>STAGE {i + 1}</b>', showarrow=False,
                           font=dict(color='#FFFFFF', size=10))
        fig.add_annotation(x=i, y=0.85, text=f'<b>{s["title"]}</b>', showarrow=False,
                           font=dict(color='#FFFFFF', size=12.5))
        fig.add_annotation(x=i, y=0.40, text='<br>'.join(s['lines']), showarrow=False, align='center',
                           font=dict(color=INK, size=11.2))
        if i < n - 1:
            fig.add_annotation(x=i + 1 - w / 2 - 0.005, y=0.5, ax=i + w / 2 + 0.005, ay=0.5, xref='x', yref='y',
                               axref='x', ayref='y', showarrow=True, arrowhead=3, arrowsize=1.1, arrowwidth=2,
                               arrowcolor=MUTED, text='')
    fig.add_trace(go.Scatter(x=list(range(n)), y=[0.45] * n, mode='markers',
                             marker=dict(size=120, opacity=0, symbol='square'),
                             hovertext=[s['hover'] for s in stages], hoverinfo='text', showlegend=False))
    fig.update_xaxes(visible=False, range=[-0.5, n - 0.5])
    fig.update_yaxes(visible=False, range=[-0.04, 1.04])
    fig.update_layout(height=360, margin=dict(l=4, r=4, t=4, b=4), plot_bgcolor='#FFFFFF',
                      paper_bgcolor='rgba(0,0,0,0)', hoverlabel=dict(bgcolor='#FFFFFF', bordercolor=BORDER,
                                                                      font=dict(color=INK, size=12)))
    return fig


with tab1:
    report = load_preprocessing_report(results_stamp())
    clean = report.get('cleaning', {})
    vsum = load_validation_summary()
    n_raw_llm, n_kept_llm = vsum if vsum else (725, 685)
    pass_rate = n_kept_llm / n_raw_llm if n_raw_llm else np.nan
    split = PCFG.get('split', {})
    split_txt = ' / '.join(f'{int(round(100 * split.get(k, 0)))}' for k in ('train', 'val', 'test'))
    seeds_txt = ', '.join(str(s) for s in MODEL_SEEDS)
    final_rows = clean.get('final_rows')
    dups = clean.get('removed_exact_duplicates')
    profiles = list(PROFILE_TEXT.keys()) or PROFILE_ORDER
    shares_txt = ' and '.join(f'{s}%' for s in sorted(RETRAINED_DIRS)) or '20% and 50%'

    stages = [
        {'title': 'Preprocessing', 'lines': [
            'CIC-IDS2017, 3 capture days',
            f'{final_rows:,} flows after cleaning' if final_rows else 'Deduplication and cleaning',
            f'{dups:,} exact duplicates removed' if dups else 'inf / NaN / negative rows removed',
            f'{split_txt} stratified split',
            f'{len(MUTABLE)} mutable vs {len(FIXED)} fixed features',
            f'StandardScaler on {len(FEATURES)} features'],
         'hover': ('<b>Stage 1 - Preprocessing</b><br>Classes: ' + ', '.join(CLASS_NAMES) +
                   f'<br>Mutable features (attacker-controllable): {len(MUTABLE)}'
                   f'<br>Fixed features (ports, flags, init windows): {len(FIXED)}')},
        {'title': 'Baseline Training', 'lines': [
            'RandomForest, MLP', '1D-CNN, LSTM', f'Seeds {seeds_txt}', 'Adam + early stopping (val loss)',
            'E1: clean held-out test set'],
         'hover': '<b>Stage 2 - Baseline Training</b><br>RF: 150 trees<br>DL: Adam lr 1e-3, batch 512, patience 5'
                  '<br>Checkpoints: models/baseline/{Model}_seed{seed}'},
        {'title': 'LLM Polymorphic Generation', 'lines': [
            f'{len(profiles)} behavioral profiles', *[p for p in profiles[:4]],
            f'{n_raw_llm} raw -> {n_kept_llm} validated ({fmt_pct(pass_rate)})', '5-step validation'],
         'hover': '<b>Stage 3 - LLM Generation</b><br>Only mutable features are rewritten; fixed features must match the'
                  ' source flow.<br>Checks: schema, range clipping, logical consistency, fixed-feature integrity,'
                  ' duplicates.'},
        {'title': 'Zero-Shot Robustness Eval', 'lines': [
            'E2: held-out LLM variants', 'E3: class-matched scripted', 'random-scaling noise control',
            'Recall drop and evasion rate', '(attack predicted as Benign)'],
         'hover': '<b>Stage 4 - Robustness Evaluation</b><br>Baseline models are tested without ever seeing'
                  ' variants.<br>E3 isolates the effect of semantics vs plain perturbation.'},
        {'title': 'Adversarial Retraining', 'lines': [
            f'Train set + {shares_txt}', 'of the variant pool (E4)', 'Recovery on held-out variants',
            'Side effects: clean F1 and FPR'],
         'hover': '<b>Stage 5 - Adversarial Retraining</b><br>Held-out variants are disjoint from the pool by'
                  ' source flow.<br>Checkpoints: models/retrained_{share}pct/'},
    ]
    section('End-to-end system architecture  (hover a stage for details)')
    st.plotly_chart(pipeline_figure(stages), config={'displayModeBar': False})

    section('Executive summary  (mean over seeds ' + seeds_txt + ')')
    models_present = [m for m in MODEL_ORDER if not np.isnan(bench_value('E1', m, 'recall'))
                      or not np.isnan(bench_value('E2', m, 'recall'))]
    if not models_present:
        st.info('No result CSVs found in the results folder yet. Run notebooks 02, 04 and 05.')
    else:
        def best(regime, metric, fn):
            vals = {m: bench_value(regime, m, metric) for m in models_present}
            vals = {m: v for m, v in vals.items() if np.isfinite(v)}
            if not vals:
                return None, np.nan
            k = fn(vals, key=vals.get)
            return k, vals[k]

        m_f1, v_f1 = best('E1', 'f1_macro', max)
        m_ev, v_ev = best('E2', 'evasion', max)
        drops = {m: bench_value('E1', m, 'recall') - bench_value('E2', m, 'recall') for m in models_present}
        drops3 = {m: bench_value('E1', m, 'recall') - bench_value('E3', m, 'recall') for m in models_present}
        excess = [drops[m] - drops3[m] for m in models_present if np.isfinite(drops[m]) and np.isfinite(drops3[m])]
        finite_drops = {m: v for m, v in drops.items() if np.isfinite(v)}
        m_rob = min(finite_drops, key=finite_drops.get) if finite_drops else None
        rec_best, rec_tag = (None, np.nan), None
        for reg in sorted({r for r in BENCH['regime'].unique() if r.startswith('E4')}, key=regime_sort_key):
            m_r, v_r = best(reg, 'recovery', max)
            if m_r is not None and (rec_tag is None or v_r > rec_best[1]):
                rec_best, rec_tag = (m_r, v_r), reg

        cards = [
            ('Clean baseline macro-F1', fmt_num(v_f1), f'Best model: {m_f1}' if m_f1 else 'E1', NAVY),
            ('Peak LLM evasion rate', fmt_pct(v_ev), f'{m_ev} on held-out variants (E2)' if m_ev else 'E2', CRIMSON),
            ('LLM excess recall drop', (f'{100 * np.mean(excess):+.1f} pp' if excess else 'n/a'),
             'E2 drop minus E3 scripted-noise drop', AMBER),
            ('Most robust architecture', m_rob or 'n/a',
             f'Smallest E2 recall drop ({fmt_num(finite_drops.get(m_rob, np.nan), signed=True)})' if m_rob else '', TEAL),
            ('Best retraining recovery', fmt_num(rec_best[1], signed=True),
             f'{rec_best[0]} at {rec_tag.replace("E4@", "")}% share' if rec_tag else 'E4', STEEL),
            ('Variant validation pass rate', fmt_pct(pass_rate), f'{n_kept_llm} of {n_raw_llm} LLM candidate flows', NAVY),
        ]
        for row in (cards[:3], cards[3:]):
            cols = st.columns(3)
            for c, (lab, val, sub, acc) in zip(cols, row):
                c.markdown(kpi_card(lab, val, sub, acc), unsafe_allow_html=True)
            st.write('')

        left, right = st.columns([1.25, 1])
        with left:
            section('Recall drop: LLM variants (E2) vs scripted noise (E3)')
            fig = go.Figure()
            fig.add_bar(name='E2 LLM variants', x=models_present, y=[drops[m] for m in models_present],
                        marker_color=CRIMSON, text=[fmt_num(drops[m]) for m in models_present], textposition='outside')
            fig.add_bar(name='E3 scripted noise', x=models_present, y=[drops3[m] for m in models_present],
                        marker_color=AMBER, text=[fmt_num(drops3[m]) for m in models_present], textposition='outside')
            fig.update_layout(barmode='group', yaxis_title='class-balanced recall drop vs E1')
            st.plotly_chart(style_fig(fig, 360), config={'displayModeBar': False})
        with right:
            section('Headline numbers per model')
            tbl = []
            for m in models_present:
                r = {'Model': m, 'E1 recall': fmt_num(bench_value('E1', m, 'recall')),
                     'E2 recall': fmt_num(bench_value('E2', m, 'recall')),
                     'E2 evasion': fmt_pct(bench_value('E2', m, 'evasion')),
                     'E3 evasion': fmt_pct(bench_value('E3', m, 'evasion'))}
                for reg in sorted({x for x in BENCH['regime'].unique() if x.startswith('E4')}, key=regime_sort_key):
                    r[f'{reg.replace("@", " @ ")}% recall'] = fmt_num(bench_value(reg, m, 'recall'))
                tbl.append(r)
            st.dataframe(pd.DataFrame(tbl), hide_index=True)
            rq = os.path.join(DIRS['results'], 'FINAL_research_questions.txt')
            if os.path.exists(rq):
                with st.expander('Research-question findings (FINAL_research_questions.txt)'):
                    with open(rq, encoding='utf-8') as fh:
                        st.text(fh.read())

# ================================================================== TAB 2
@st.cache_data(show_spinner=False)
def compute_projection(attack_class, method, n_bg, seed=42):
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import NearestNeighbors
    tr = load_train_raw()
    var, _ = load_llm_variants()
    mut = [f for f in MUTABLE if f in tr.columns]
    rng = np.random.default_rng(seed)

    ben_pos = np.flatnonzero(tr['Label'].to_numpy() == BENIGN)
    att_pos = np.flatnonzero(tr['Label'].to_numpy() == attack_class)
    ben_s = rng.choice(ben_pos, size=min(n_bg, len(ben_pos)), replace=False)
    att_s = rng.choice(att_pos, size=min(n_bg, len(att_pos)), replace=False)
    v = var[var['Label'] == attack_class].reset_index(drop=True)
    sid = v['source_row_id'].to_numpy()
    valid_sid = np.isfinite(sid) & (sid >= 0) & (sid < len(tr))
    src_rows = np.unique(sid[valid_sid].astype(int))
    att_s = np.union1d(att_s, src_rows)

    real = pd.concat([tr.iloc[ben_s][mut].assign(_group='Benign Traffic', _profile='', _rid=ben_s),
                      tr.iloc[att_s][mut].assign(_group='Original Attack Flows', _profile='', _rid=att_s)],
                     ignore_index=True)
    vv = v[mut].assign(_group='LLM Polymorphic Variants', _profile=v['style'].to_numpy(),
                       _rid=np.where(valid_sid, sid, -1).astype(int))
    allf = pd.concat([real, vv], ignore_index=True)
    Z = np.log1p(np.clip(allf[mut].to_numpy(dtype=np.float64), 0, None))
    Zs = StandardScaler().fit_transform(Z)
    info = ''
    if method == 'PCA':
        from sklearn.decomposition import PCA
        p = PCA(n_components=2, random_state=seed)
        emb = p.fit_transform(Zs)
        ev = p.explained_variance_ratio_
        info = f'PC1 {100 * ev[0]:.1f}% and PC2 {100 * ev[1]:.1f}% of variance'
    else:
        import umap
        emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=seed).fit_transform(Zs)
        info = 'UMAP, 15 neighbours, min_dist 0.1'

    out = allf[['_group', '_profile', '_rid']].copy()
    out['x'], out['y'] = emb[:, 0], emb[:, 1]
    out['variant_idx'] = -1
    out.loc[out['_group'] == 'LLM Polymorphic Variants', 'variant_idx'] = np.arange(len(v))

    n_real = len(real)
    is_att_real = (real['_group'] == 'Original Attack Flows').to_numpy()
    nn = NearestNeighbors(n_neighbors=1).fit(Zs[:n_real])
    _, idx = nn.kneighbors(Zs[n_real:])
    frac_attack_nn = float(is_att_real[idx[:, 0]].mean()) if len(v) else np.nan
    o = out[out['_group'] == 'Original Attack Flows']
    lo_x, hi_x = np.percentile(o['x'], [5, 95])
    lo_y, hi_y = np.percentile(o['y'], [5, 95])
    vo = out[out['_group'] == 'LLM Polymorphic Variants']
    outside = ((vo['x'] < lo_x) | (vo['x'] > hi_x) | (vo['y'] < lo_y) | (vo['y'] > hi_y)).to_numpy()
    stats = {'n_benign': int(len(ben_s)), 'n_orig': int(len(att_s)), 'n_var': int(len(v)),
             'frac_attack_nn': frac_attack_nn, 'frac_outside': float(outside.mean()) if len(vo) else np.nan,
             'box': (lo_x, hi_x, lo_y, hi_y), 'info': info, 'n_mut': len(mut)}
    return out, stats


with tab2:
    if TRAIN_RAW is None or VARIANTS is None:
        st.warning('train_raw.parquet and the LLM variant file are both required for the projection.')
    else:
        c1, c2, c3, c4 = st.columns([1, 1, 1.4, 1.4])
        avail_cls = [c for c in ATTACK_CLASSES if (VARIANTS['Label'] == c).any()]
        p_cls = c1.selectbox('Attack Class', avail_cls, key='proj_cls')
        p_method = c2.radio('Projection Method', ['PCA', 'UMAP'], horizontal=True, key='proj_method')
        p_n = c3.slider('Background sample size (per real group)', 500, 6000, 2000, 500, key='proj_n')
        p_color = c4.radio('Color LLM variants by', ['Behavioral Profile', 'Unified LLM Group'], horizontal=True,
                           key='proj_color')
        show_links = st.checkbox('Draw displacement vectors from each source flow to its LLM variant', value=False)

        try:
            with st.spinner(f'Computing {p_method} projection on log1p mutable features ...'):
                proj, pstats = compute_projection(p_cls, p_method, p_n)
        except ImportError as exc:
            st.error(f'{p_method} dependency missing: {exc}. Install umap-learn in Cell 1.')
            proj = None
        if proj is not None:
            fig = go.Figure()
            b = proj[proj['_group'] == 'Benign Traffic']
            o = proj[proj['_group'] == 'Original Attack Flows']
            v = proj[proj['_group'] == 'LLM Polymorphic Variants']
            fig.add_trace(go.Scattergl(x=b['x'], y=b['y'], mode='markers', name='Benign Traffic',
                                       marker=dict(size=4, color=SLATE, opacity=0.35),
                                       hovertemplate='Benign Traffic<br>train row %{customdata}<extra></extra>',
                                       customdata=b['_rid']))
            fig.add_trace(go.Scattergl(x=o['x'], y=o['y'], mode='markers', name=f'Original {p_cls} Flows',
                                       marker=dict(size=5, color=NAVY, opacity=0.55),
                                       hovertemplate='Original attack flow<br>train row %{customdata}<extra></extra>',
                                       customdata=o['_rid']))
            if show_links and len(v):
                pos = dict(zip(o['_rid'].to_numpy(), zip(o['x'].to_numpy(), o['y'].to_numpy())))
                lx, ly = [], []
                for rid, x, y in zip(v['_rid'], v['x'], v['y']):
                    if rid in pos:
                        lx += [pos[rid][0], x, None]
                        ly += [pos[rid][1], y, None]
                fig.add_trace(go.Scattergl(x=lx, y=ly, mode='lines', name='Source -> variant shift',
                                           line=dict(color=rgba(CRIMSON, 0.25), width=1), hoverinfo='skip'))
            if p_color == 'Behavioral Profile':
                for prof in [p for p in PROFILE_ORDER if p in set(v['_profile'])] + \
                            sorted(set(v['_profile']) - set(PROFILE_ORDER)):
                    vp = v[v['_profile'] == prof]
                    fig.add_trace(go.Scattergl(
                        x=vp['x'], y=vp['y'], mode='markers', name=f'LLM: {prof}',
                        marker=dict(size=8, color=PROFILE_COLORS.get(prof, CRIMSON), symbol='diamond',
                                    line=dict(color='#FFFFFF', width=0.8)),
                        customdata=np.stack([vp['_rid'], vp['variant_idx']], axis=1),
                        hovertemplate=f'LLM variant ({prof})<br>source train row %{{customdata[0]}}<extra></extra>'))
            else:
                fig.add_trace(go.Scattergl(
                    x=v['x'], y=v['y'], mode='markers', name='LLM Polymorphic Variants',
                    marker=dict(size=8, color=CRIMSON, symbol='diamond', line=dict(color='#FFFFFF', width=0.8)),
                    customdata=v['_rid'], hovertemplate='LLM variant<br>source train row %{customdata}<extra></extra>'))
            lo_x, hi_x, lo_y, hi_y = pstats['box']
            fig.add_shape(type='rect', x0=lo_x, x1=hi_x, y0=lo_y, y1=hi_y, line=dict(color=NAVY, width=1, dash='dot'),
                          fillcolor='rgba(0,0,0,0)')
            fig.add_annotation(x=hi_x, y=hi_y, text='original-attack 5-95% envelope', showarrow=False,
                               xanchor='right', yanchor='bottom', font=dict(size=10, color=NAVY),
                               bgcolor='rgba(255,255,255,0.85)')
            ax_lab = ('PC1', 'PC2') if p_method == 'PCA' else ('UMAP-1', 'UMAP-2')
            fig.update_layout(xaxis_title=ax_lab[0], yaxis_title=ax_lab[1])
            st.plotly_chart(style_fig(fig, 600))

            k1, k2, k3, k4 = st.columns(4)
            k1.markdown(kpi_card('Points plotted', f'{pstats["n_benign"] + pstats["n_orig"] + pstats["n_var"]:,}',
                                 f'{pstats["n_benign"]:,} benign / {pstats["n_orig"]:,} original / {pstats["n_var"]} LLM',
                                 NAVY), unsafe_allow_html=True)
            k2.markdown(kpi_card('Outside original envelope', fmt_pct(pstats['frac_outside']),
                                 'LLM variants beyond the 5-95% box of the original class', CRIMSON),
                        unsafe_allow_html=True)
            k3.markdown(kpi_card('Nearest real neighbour = attack', fmt_pct(pstats['frac_attack_nn']),
                                 f'in the full {pstats["n_mut"]}-D log1p space (manifold check)', TEAL),
                        unsafe_allow_html=True)
            k4.markdown(kpi_card('Projection', p_method, pstats['info'], STEEL), unsafe_allow_html=True)
            st.markdown(
                '<div class="callout">Each point is a flow described by its log1p-transformed, standardised mutable '
                'features (the only features the LLM was allowed to change). Variants that leave the dotted envelope '
                'move away from the axis-aligned regions where tree and threshold-like boundaries were fitted, while a '
                'high nearest-neighbour-attack share shows they still stay on the attack manifold instead of collapsing '
                'into benign traffic.</div>', unsafe_allow_html=True)

# ================================================================== TAB 3
with tab3:
    if BENCH.empty:
        st.warning('No benchmark CSVs were found in the results folder. Run notebooks 02, 04 and 05.')
    else:
        regimes_present = sorted(BENCH['regime'].unique(), key=regime_sort_key)
        all_regimes = sorted(set(regimes_present) | {'E1', 'E2', 'E3', 'E4@20', 'E4@50'}, key=regime_sort_key)
        c1, c2, c3 = st.columns([1.2, 1.4, 1])
        scope = c1.selectbox('Attack Class Scope', ['All Attack Classes (Macro Summary)'] + ATTACK_CLASSES)
        reg_choice = c2.selectbox('Evaluation Regime', ['All Regimes Side-by-Side'] + [regime_label(r) for r in all_regimes])
        metric_lab = c3.radio('Metric', ['Attack recall', 'Evasion rate'], horizontal=True)
        metric = 'recall' if metric_lab == 'Attack recall' else 'evasion'
        cls_key = 'ALL' if scope.startswith('All') else scope
        sub = BENCH[(BENCH['class'] == cls_key) & (BENCH['metric'] == metric)]
        models_b = [m for m in MODEL_ORDER if m in set(BENCH['model'])]
        y_title = ('class-balanced attack recall' if cls_key == 'ALL' else f'{cls_key} recall') if metric == 'recall' \
            else ('evasion rate (attack predicted Benign)')

        def val(reg, m, frame=sub):
            r = frame[(frame['regime'] == reg) & (frame['model'] == m)]
            return (float(r['mean'].iloc[0]), float(r['std'].iloc[0])) if len(r) else (np.nan, np.nan)

        fig = go.Figure()
        if reg_choice == 'All Regimes Side-by-Side':
            for reg in regimes_present:
                vals = [val(reg, m) for m in models_b]
                if all(np.isnan(a) for a, _ in vals):
                    continue
                fig.add_bar(name=regime_label(reg), x=models_b, y=[a for a, _ in vals],
                            error_y=dict(type='data', array=[s for _, s in vals], color=MUTED, thickness=1),
                            marker_color=REGIME_COLORS.get(reg, SLATE),
                            text=[fmt_num(a) for a, _ in vals], textposition='outside', textfont=dict(size=10))
            fig.update_layout(barmode='group')
        else:
            reg = next(r for r in all_regimes if regime_label(r) == reg_choice)
            vals = [val(reg, m) for m in models_b]
            if all(np.isnan(a) for a, _ in vals):
                st.info(f'No {metric_lab.lower()} values are stored for {reg_choice} and scope "{scope}".')
            fig.add_bar(name=regime_label(reg), x=models_b, y=[a for a, _ in vals],
                        error_y=dict(type='data', array=[s for _, s in vals], color=MUTED, thickness=1),
                        marker_color=[MODEL_COLORS[m] for m in models_b],
                        text=[fmt_num(a) for a, _ in vals], textposition='outside')
            if reg != 'E1':
                ref = [val('E1', m)[0] for m in models_b]
                fig.add_trace(go.Scatter(x=models_b, y=ref, mode='markers', name='E1 reference',
                                         marker=dict(symbol='line-ew', size=34, line=dict(color=INK, width=2.5))))
        top = np.nanmax([*sub['mean'].to_numpy(), 0.05])
        fig.update_layout(yaxis_title=y_title, yaxis_range=[0, min(1.12, top * 1.18) if metric == 'evasion' else 1.12])
        st.plotly_chart(style_fig(fig, 440))

        section(f'Comparison table  ({metric_lab.lower()}, mean{PM.strip()}std over seeds)')
        show_regs = regimes_present if reg_choice == 'All Regimes Side-by-Side' else \
            [r for r in all_regimes if regime_label(r) == reg_choice]
        tab_rows = []
        for m in models_b:
            r = {'Model': m}
            for reg in show_regs:
                a, s = val(reg, m)
                r[regime_label(reg)] = 'n/a' if np.isnan(a) else f'{a:.4f}{PM}{s:.4f}'
            if cls_key == 'ALL':
                for reg in show_regs:
                    for met, lab in [('f1_macro', 'clean F1'), ('fpr', 'clean FPR'), ('recovery', 'recovery')]:
                        a, s = val(reg, m, BENCH[(BENCH['class'] == 'ALL') & (BENCH['metric'] == met)])
                        if not np.isnan(a):
                            r[f'{reg} {lab}'] = f'{a:.4f}{PM}{s:.4f}'
            tab_rows.append(r)
        st.dataframe(pd.DataFrame(tab_rows), hide_index=True)
        st.caption('Sources: ' + ', '.join(BENCH_SOURCES) + '. E2 and E4 use the held-out LLM variant set that no '
                   'retrained model has seen; E1 uses the attack flows of the original test split.')

        by_style = results_csv('04_E2_by_style.csv')
        if by_style is not None and {'model', 'style', 'recall', 'evasion'} <= set(by_style.columns):
            section('E2 breakdown by LLM behavioral profile (04_E2_by_style.csv)')
            fig = go.Figure()
            styles = [p for p in PROFILE_ORDER if p in set(by_style['style'])] + \
                sorted(set(by_style['style']) - set(PROFILE_ORDER))
            for m in [x for x in MODEL_ORDER if x in set(by_style['model'])]:
                d = by_style[by_style['model'] == m].set_index('style').reindex(styles)
                fig.add_bar(name=m, x=styles, y=d[metric], marker_color=MODEL_COLORS[m],
                            text=[fmt_num(x) for x in d[metric]], textposition='outside', textfont=dict(size=10))
            fig.update_layout(barmode='group', yaxis_title=y_title)
            st.plotly_chart(style_fig(fig, 380))

        section('FINAL_summary_table.csv')
        if FINAL is not None:
            st.dataframe(FINAL)
        else:
            st.info('FINAL_summary_table.csv not found.')

        with st.expander('All result CSV files discovered in the results folder'):
            files = sorted(glob.glob(os.path.join(DIRS['results'], '*.csv')),
                           key=lambda f: (not re.search(r'per_class|by_style|per_profile', os.path.basename(f)),
                                          os.path.basename(f)))
            if files:
                pick = st.selectbox('Preview file', [os.path.basename(f) for f in files])
                st.dataframe(results_csv(pick))
            else:
                st.write('No CSV files found.')

        with st.expander('Figure gallery (results/figures)', expanded=False):
            figs = [('fig1_original_vs_variants.png', 'Fig. 1 - Original data vs variants'),
                    ('fig2_evasion_rate.png', 'Fig. 2 - Evasion rate per attack type'),
                    ('fig3_before_after_retraining.png', 'Fig. 3 - Before vs after adversarial retraining')]
            for fname, cap in figs:
                p = os.path.join(DIRS['figures'], fname)
                if os.path.exists(p):
                    st.image(p, caption=cap)
                else:
                    st.caption(f'{fname} not found')
            extra = sorted(set(glob.glob(os.path.join(DIRS['figures'], '*.png')) +
                               glob.glob(os.path.join(DIRS['results'], '*.png'))) -
                           {os.path.join(DIRS['figures'], f) for f, _ in figs})
            if extra:
                pick = st.selectbox('Additional figures', ['(none)'] + [os.path.relpath(f, DIRS['results']) for f in extra])
                if pick != '(none)':
                    st.image(os.path.join(DIRS['results'], pick), caption=pick)

# ================================================================== TAB 4
@st.cache_data(show_spinner='Scoring every LLM variant with the selected checkpoints ...')
def variant_predictions(model_dir, seed):
    var, _ = load_llm_variants()
    models, _ = load_model_set(model_dir, seed)
    X = load_scaler().transform(var[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
    return pd.DataFrame({name: predict_proba(name, m, X).argmax(axis=1) for name, m in models.items()}, index=var.index)


def model_card(name, proba, true_id, mode, base_pred=None):
    pred = int(np.argmax(proba))
    conf = float(proba[pred])
    p_true = float(proba[true_id])
    is_attack = pred != BENIGN_ID
    if mode == 'original':
        kind, badge = ('ok', 'DETECTED') if is_attack else ('bad', 'MISSED')
    elif mode == 'baseline':
        kind, badge = ('ok', 'CAUGHT') if is_attack else ('bad', 'EVADED')
    else:
        base_evaded = base_pred is not None and base_pred == BENIGN_ID
        if is_attack:
            kind, badge = ('ok', 'RECOVERED') if base_evaded else ('ok', 'CAUGHT')
        else:
            kind, badge = ('bad', 'MISSED')
    pred_txt = CLASS_NAMES[pred]
    if pred == BENIGN_ID:
        verdict = 'Predicted Benign'
    elif pred == true_id:
        verdict = 'correct attack class'
    else:
        verdict = f'flagged as attack (class {CLASS_NAMES[pred]})'
    bar_col = OK_GREEN if kind == 'ok' else BAD_RED
    return (f'<div class="model-card {kind}"><div class="mc-head"><span class="mc-name">{esc(name)}</span>'
            f'{pill(badge, kind)}</div>'
            f'<div class="mc-row">Predicted <b>{esc(pred_txt)}</b> &nbsp;|&nbsp; {esc(verdict)}</div>'
            f'<div class="mc-row">Confidence <b>{fmt_pct(conf)}</b> &nbsp;|&nbsp; P({esc(CLASS_NAMES[true_id])}) '
            f'<b>{fmt_pct(p_true)}</b></div>'
            f'<div class="conf-track"><div class="conf-fill" style="width:{100 * conf:.1f}%;background:{bar_col}"></div></div>'
            '</div>')


def multiplier(o, v):
    if o == 0 and v == 0:
        return 1.0
    if o == 0:
        return np.nan
    return v / o


with tab4:
    ready = TRAIN_RAW is not None and SCALER is not None and VARIANTS is not None and IC is not None
    if not ready:
        st.warning('The simulator needs train_raw.parquet, scaler.joblib, the LLM variants and code/ids_common.py.')
    elif not BASE_MODELS:
        st.warning(f'No baseline checkpoints could be loaded for seed {SEED} from {DIRS["baseline_models"]}.')
    else:
        base_preds = variant_predictions(DIRS['baseline_models'], SEED)
        ret_preds = variant_predictions(RETRAINED_DIR, SEED) if RET_MODELS else None
        c1, c2, c3 = st.columns([1, 1.4, 1.2])
        s_cls = c1.selectbox('Attack Class', [c for c in ATTACK_CLASSES if (VARIANTS['Label'] == c).any()], key='sim_cls')
        cls_df = VARIANTS[VARIANTS['Label'] == s_cls]
        profs = [p for p in PROFILE_ORDER if p in set(cls_df['style'])] + sorted(set(cls_df['style']) - set(PROFILE_ORDER))
        s_prof = c2.selectbox('LLM Behavioral Profile', profs, key='sim_prof')
        only_evasive = c3.checkbox('Only variants that evade at least one baseline model', value=False)
        pool = cls_df[cls_df['style'] == s_prof]
        if only_evasive:
            pool = pool[(base_preds.loc[pool.index] == BENIGN_ID).any(axis=1)]
        if s_prof in PROFILE_TEXT:
            st.markdown(f'<div class="callout"><b>Profile instruction given to the LLM:</b> {esc(PROFILE_TEXT[s_prof])}</div>',
                        unsafe_allow_html=True)
        if pool.empty:
            st.info('No variants match this selection. Clear the evasive-only filter or pick another profile.')
        else:
            s_idx = (st.slider('Variant Sample Index', 1, len(pool), 1, key=f'sim_idx_{s_cls}_{s_prof}_{only_evasive}') - 1) \
                if len(pool) > 1 else 0
            vrow = pool.iloc[s_idx]
            vidx = pool.index[s_idx]
            true_id = LABEL_TO_ID[s_cls]
            sid = vrow['source_row_id']
            has_src = np.isfinite(sid) and 0 <= int(sid) < len(TRAIN_RAW)
            sel = load_augmentation_selection()
            used = ''
            if SHARE and 'variant_id' in VARIANTS.columns and f'{SHARE}pct' in sel:
                used = 'yes' if vrow['variant_id'] in sel[f'{SHARE}pct'] else 'no'
            meta = [('Variant', f'{s_idx + 1} of {len(pool)}'),
                    ('Variant id', vrow['variant_id'] if 'variant_id' in VARIANTS.columns else 'n/a'),
                    ('Evaluation split', vrow['split']),
                    ('Source train row', int(sid) if has_src else 'n/a'),
                    ('Generator', vrow['generator'] if 'generator' in VARIANTS.columns else 'n/a')]
            if used:
                meta.append((f'Used in E4 @ {SHARE}% training', used))
            st.markdown('<div class="meta-strip">' + ''.join(f'<span>{esc(k)}: <b>{esc(v)}</b></span>' for k, v in meta)
                        + '</div>', unsafe_allow_html=True)
            if used == 'yes':
                st.caption('This variant was part of the retraining set at the selected share, so the retrained models have '
                           'seen it. Pick a held-out variant for an unbiased view of recovery.')

            # ---------------- step 1
            st.markdown('#### Step 1 - Original attack flow -> 4 baseline models')
            if not has_src:
                st.info('This variant has no valid source_row_id, so the original flow cannot be retrieved.')
                orig = None
            else:
                orig = TRAIN_RAW.iloc[int(sid)]
                if orig['Label'] != s_cls:
                    st.caption(f'Note: the source row is labelled {orig["Label"]} in train_raw.parquet.')
                Xo = SCALER.transform(orig[FEATURES].to_numpy(dtype=np.float32).reshape(1, -1)).astype(np.float32)
                cols = st.columns(len(MODEL_ORDER))
                for col, name in zip(cols, MODEL_ORDER):
                    if name in BASE_MODELS:
                        col.markdown(model_card(name, predict_proba(name, BASE_MODELS[name], Xo)[0], true_id, 'original'),
                                     unsafe_allow_html=True)
                    else:
                        col.markdown(f'<div class="model-card neutral"><div class="mc-head"><span class="mc-name">{esc(name)}'
                                     f'</span>{pill("MISSING", "neutral")}</div></div>', unsafe_allow_html=True)

            # ---------------- step 2
            st.markdown('#### Step 2 - LLM polymorphic feature shift')
            if orig is not None:
                view = st.radio('Features shown', ['Key behavioral features', 'Top 12 most-shifted mutable features'],
                                horizontal=True, key='sim_feat_view')
                recs = []
                for f in MUTABLE:
                    o_v, n_v = float(orig[f]), float(vrow[f])
                    mlt = multiplier(o_v, n_v)
                    recs.append({'Feature': f, 'Original': o_v, 'LLM variant': n_v, 'Multiplier': mlt,
                                 'abs_log': abs(np.log(mlt)) if np.isfinite(mlt) and mlt > 0 else
                                 (np.inf if n_v != o_v else 0.0)})
                shift = pd.DataFrame(recs)
                if view.startswith('Key'):
                    shown = shift[shift['Feature'].isin(KEY_FEATURES)].set_index('Feature').reindex(
                        [f for f in KEY_FEATURES if f in set(shift['Feature'])]).reset_index()
                else:
                    shown = shift.sort_values('abs_log', ascending=False).head(12)
                left, right = st.columns([1.05, 1])
                with left:
                    t = shown[['Feature', 'Original', 'LLM variant', 'Multiplier']].copy()
                    t['Original'] = t['Original'].map(lambda x: f'{x:,.2f}')
                    t['LLM variant'] = t['LLM variant'].map(lambda x: f'{x:,.2f}')
                    t['Multiplier'] = t['Multiplier'].map(lambda x: 'new (orig 0)' if not np.isfinite(x) else f'x{x:,.3f}')
                    st.dataframe(t, hide_index=True)
                    fixed_ok = all(np.isclose(float(orig[f]), float(vrow[f]), rtol=1e-5, atol=1e-6) for f in FIXED)
                    n_changed = int((shift['abs_log'] > 1e-9).sum())
                    st.markdown(f'{pill("FIXED FEATURES INTACT", "ok") if fixed_ok else pill("FIXED FEATURES CHANGED", "bad")}'
                                f' &nbsp; <span style="font-size:0.85rem;color:#475569">{n_changed} of {len(MUTABLE)} '
                                'mutable features changed</span>', unsafe_allow_html=True)
                with right:
                    ch = shown[np.isfinite(shown['Multiplier']) & (shown['Multiplier'] > 0)].iloc[::-1]
                    lg = np.log10(ch['Multiplier'].to_numpy(dtype=float))
                    fig = go.Figure(go.Bar(
                        x=lg, y=ch['Feature'], orientation='h', cliponaxis=False,
                        marker_color=[AMBER if m >= 1 else STEEL for m in ch['Multiplier']],
                        text=[f'x{m:,.3g}' for m in ch['Multiplier']], textposition='outside',
                        customdata=ch['Multiplier'], hovertemplate='%{y}<br>multiplier %{customdata:.4g}<extra></extra>'))
                    fig.add_vline(x=0, line=dict(color=INK, width=1))
                    span = max(float(lg.max(initial=0)) - float(lg.min(initial=0)), 1.0)
                    lo = float(min(lg.min(initial=0), 0)) - 0.22 * span - 0.3
                    hi = float(max(lg.max(initial=0), 0)) + 0.22 * span + 0.3
                    step = max(1, int(np.ceil((hi - lo) / 7)))
                    ticks = [t for t in range(int(np.floor(lo)), int(np.ceil(hi)) + 1) if t % step == 0]
                    fig.update_xaxes(range=[lo, hi], tickvals=ticks, ticktext=[f'x{10.0 ** t:g}' for t in ticks],
                                     title='variant / original (log10 scale, centred at x1)')
                    fig.update_layout(title=dict(text='Relative multiplier (amber = increased, blue = decreased)',
                                                 font=dict(size=12)), showlegend=False)
                    st.plotly_chart(style_fig(fig, 420), config={'displayModeBar': False})

            # ---------------- step 3
            st.markdown('#### Step 3 - Mutated variant -> baseline models vs retrained models')
            Xv = SCALER.transform(vrow[FEATURES].to_numpy(dtype=np.float32).reshape(1, -1)).astype(np.float32)
            left, right = st.columns(2)
            base_out, ret_out = {}, {}
            with left:
                st.markdown('<div class="panel-title">Before retraining - baseline models (E1 checkpoints)</div>'
                            f'<div class="panel-sub">seed {SEED}, never trained on any LLM variant</div>',
                            unsafe_allow_html=True)
                for name in MODEL_ORDER:
                    if name in BASE_MODELS:
                        p = predict_proba(name, BASE_MODELS[name], Xv)[0]
                        base_out[name] = p
                        st.markdown(model_card(name, p, true_id, 'baseline'), unsafe_allow_html=True)
            with right:
                st.markdown(f'<div class="panel-title">After adversarial retraining - defended models '
                            f'(E4 @ {SHARE}%)</div><div class="panel-sub">seed {SEED}, trained with '
                            f'{SHARE}% of the variant pool added</div>' if SHARE else
                            '<div class="panel-title">After adversarial retraining</div>', unsafe_allow_html=True)
                if not RET_MODELS:
                    st.info('No retrained checkpoints were found under models/retrained_*pct.')
                for name in MODEL_ORDER:
                    if name in RET_MODELS:
                        p = predict_proba(name, RET_MODELS[name], Xv)[0]
                        ret_out[name] = p
                        bp = int(np.argmax(base_out[name])) if name in base_out else None
                        st.markdown(model_card(name, p, true_id, 'retrained', bp), unsafe_allow_html=True)

            if base_out:
                n_ev = sum(int(np.argmax(p)) == BENIGN_ID for p in base_out.values())
                rec = sum(1 for n, p in ret_out.items() if int(np.argmax(p)) != BENIGN_ID and n in base_out
                          and int(np.argmax(base_out[n])) == BENIGN_ID)
                miss = sum(int(np.argmax(p)) == BENIGN_ID for p in ret_out.values())
                k1, k2, k3 = st.columns(3)
                k1.markdown(kpi_card('Baseline models evaded', f'{n_ev} of {len(base_out)}',
                                     'variant predicted as Benign', CRIMSON), unsafe_allow_html=True)
                k2.markdown(kpi_card('Recovered after retraining', f'{rec} of {n_ev}' if n_ev else '0 of 0',
                                     'evaded before, caught after', OK_GREEN), unsafe_allow_html=True)
                k3.markdown(kpi_card('Still missed after retraining', f'{miss} of {len(ret_out)}' if ret_out else 'n/a',
                                     'retrained model predicts Benign', AMBER), unsafe_allow_html=True)

                fig = go.Figure()
                names = [n for n in MODEL_ORDER if n in base_out]
                fig.add_bar(name='Baseline (E1 checkpoint)', x=names, y=[base_out[n][true_id] for n in names],
                            marker_color=CRIMSON, text=[fmt_pct(base_out[n][true_id]) for n in names], textposition='outside')
                if ret_out:
                    fig.add_bar(name=f'Retrained (E4 @ {SHARE}%)', x=names,
                                y=[ret_out[n][true_id] if n in ret_out else np.nan for n in names], marker_color=TEAL,
                                text=[fmt_pct(ret_out[n][true_id]) if n in ret_out else '' for n in names],
                                textposition='outside')
                fig.update_layout(barmode='group', yaxis_title=f'P({s_cls}) assigned to the variant', yaxis_range=[0, 1.12])
                st.plotly_chart(style_fig(fig, 340), config={'displayModeBar': False})

            with st.expander(f'Evasion overview for all {s_cls} variants (seed {SEED})'):
                cls_idx = VARIANTS.index[VARIANTS['Label'] == s_cls]
                rows = []
                for prof in profs:
                    ix = cls_idx[VARIANTS.loc[cls_idx, 'style'] == prof]
                    r = {'Profile': prof, 'Variants': len(ix)}
                    for n in base_preds.columns:
                        r[f'{n} baseline evasion'] = fmt_pct(float((base_preds.loc[ix, n] == BENIGN_ID).mean()))
                        if ret_preds is not None and n in ret_preds.columns:
                            r[f'{n} E4 evasion'] = fmt_pct(float((ret_preds.loc[ix, n] == BENIGN_ID).mean()))
                    rows.append(r)
                st.dataframe(pd.DataFrame(rows), hide_index=True)
                st.caption('Computed live over every variant of this class (retraining-pool and held-out variants together), '
                           'so it can differ from the held-out-only numbers in tab 3.')

## Cell 3 - Launch Streamlit and open the dashboard in a new tab

Starts Streamlit headless on port 8501, opens a Cloudflare Quick Tunnel, parses the `https://*.trycloudflare.com`
address, shows a launch button, and asks the browser to open it in a new tab. If the browser blocks the pop-up,
use the button. A fresh tunnel can need 10-20 seconds before it answers; reload the page if it shows a Cloudflare error.
Logs: `/content/streamlit.log` and `/content/cloudflared.log`.

In [ ]:
import os, re, time, html, subprocess, urllib.request
from IPython.display import HTML, display

BASE_DIR = globals().get('BASE_DIR', '/content/drive/MyDrive/IDS_Polymorphic_Project')
CLOUDFLARED = globals().get('CLOUDFLARED', '/content/cloudflared')
os.chdir('/content')
PORT = 8501

# stop any previous run of this notebook (pkill excludes itself, no shell involved)
for pattern in ['streamlit run app.py', 'cloudflared tunnel --url']:
    subprocess.run(['pkill', '-f', pattern])
time.sleep(1.5)

env = dict(os.environ, IDS_BASE_DIR=BASE_DIR)
st_log = open('/content/streamlit.log', 'w')
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', str(PORT), '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
     '--theme.base', 'light', '--theme.primaryColor', '#1E3A8A'],
    stdout=st_log, stderr=subprocess.STDOUT, env=env)

healthy = False
for _ in range(90):
    if streamlit_proc.poll() is not None:
        break
    try:
        healthy = urllib.request.urlopen(f'http://localhost:{PORT}/_stcore/health', timeout=2).read().strip() == b'ok'
    except Exception:
        healthy = False
    if healthy:
        break
    time.sleep(1)
if not healthy:
    print(open('/content/streamlit.log').read()[-3000:])
    raise RuntimeError('Streamlit did not start; see the log above.')
print(f'Streamlit is running on http://localhost:{PORT}')

cf_log_path = '/content/cloudflared.log'
cf_proc = subprocess.Popen([CLOUDFLARED, 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
                           stdout=open(cf_log_path, 'w'), stderr=subprocess.STDOUT)
public_url = None
for _ in range(120):
    time.sleep(1)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open(cf_log_path).read())
    if m:
        public_url = m.group(0)
        break
    if cf_proc.poll() is not None:
        break
if not public_url:
    print(open(cf_log_path).read()[-3000:])
    raise RuntimeError('Could not obtain a trycloudflare.com address; see the log above.')
time.sleep(4)   # let the tunnel register before the browser connects

u = html.escape(public_url)
display(HTML(f'''
<div style="font-family:Inter,system-ui,sans-serif;border:1px solid #E2E8F0;border-left:5px solid #1E3A8A;
            border-radius:12px;padding:18px 22px;margin:8px 0;background:#F8FAFC;max-width:760px">
  <div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:#64748B;font-weight:600">
    Dashboard is live</div>
  <div style="font-size:18px;font-weight:700;color:#0F172A;margin:6px 0 12px 0">
    Deep Learning IDS Robustness Against LLM-Generated Polymorphic Attacks</div>
  <a href="{u}" target="_blank" rel="noopener"
     style="display:inline-block;background:#1E3A8A;color:#FFFFFF;text-decoration:none;font-weight:600;
            padding:10px 22px;border-radius:8px;font-size:14px">Open dashboard in a new tab</a>
  <div style="font-size:12px;color:#64748B;margin-top:10px">{u}</div>
</div>'''))

try:
    from google.colab import output
    output.eval_js(f'window.open("{public_url}", "_blank")')
except Exception as exc:
    print('Automatic tab opening was blocked or unavailable; use the button above.', exc)

print('To stop the dashboard: streamlit_proc.terminate(); cf_proc.terminate()')